# Introducción

Este proyecto utiliza la **API-FOOTBALL v3**, una API REST con información actualizada e histórica de más de **1100 ligas y copas** (resultados, alineaciones, eventos, clasificaciones y estadísticas).  
El acceso se realiza mediante solicitudes **HTTP autenticadas con API key**, con filtros por **fecha, liga, equipo o temporada**.

En este notebook se documenta y ejecuta el **pipeline ETL orquestado con Prefect**, cuya lógica reside en `scripts/etl_fixtures.py`.  
Se muestra el **setup**, la **ejecución del flow**, y cómo **servir y monitorear** el flujo desde **Prefect Cloud**.

El pipeline persiste datos en un **Data Lake estructurado en capas Delta Lake** (**Bronze → Silver → Gold**) y genera **exportables en CSV y Parquet**.

Además, el flujo puede ejecutarse:
- 🧩 **Manualmente** desde este notebook, para validaciones o cargas controladas.  
- 🔁 **Automáticamente**, programado una vez al día mediante Prefect (cron diario a las 06:00 UTC).


## Uso del flujo desde `scripts/etl_fixtures.py`

El flujo ETL está definido en `scripts/etl_fixtures.py`.  
En este notebook es posible:

- **Opción A — Corrida única (demo one-off):** ejecutar el flow una sola vez para validar la orquestación.  
- **Opción B — Servir el flow (opcional):** mantener el flow activo como servicio local (con programación automática).


In [1]:
import prefect
print("Prefect versión:", prefect.__version__)

Prefect versión: 2.20.9


### ⭐ Opción A — Corrida manual del flujo (one-off)

Ejecuta el flujo `etl_api_football` **una sola vez**, ideal para validar que la orquestación funciona correctamente. Permite comprobar:

- que la API-Football responde con datos válidos  
- que las transformaciones Bronze → Silver → Gold se ejecutan sin errores  
- que el flujo persiste correctamente las particiones en Delta Lake  

Es la opción recomendada para pruebas locales, validaciones rápidas y depuración antes de activar cualquier programación automática.


In [2]:
import importlib

# Importa el módulo de orquestación
etl = importlib.import_module("scripts.etl_fixtures")

# Ejecuta el flujo ETL de forma local (modo recomendado dentro del Notebook)
etl.etl_api_football(endpoints=["fixtures"])

# Nota:
# Este mismo flujo puede ejecutarse desde la terminal:
#     python scripts/etl_fixtures.py
#
# O desde el notebook:
#     !python scripts/etl_fixtures.py
#
# Todas las opciones ejecutan exactamente el mismo flow.

06:41:52.542 | INFO    | prefect.engine - Created flow run 'cautious-badger' for flow 'etl-api-football'

06:41:52.555 | INFO    | Flow run 'cautious-badger' - View at https://app.prefect.cloud/account/1513bf29-3686-40b8-9dbf-c85ba6a6f8c0/workspace/377710aa-6343-48a1-a52d-8fd9be6fbba7/flow-runs/flow-run/0692c116-27ad-7fb7-8000-073d519a952d

06:41:53.213 | INFO    | Flow run 'cautious-badger' - Created task run 'task_extract-0' for task 'task_extract'

06:41:53.229 | INFO    | Flow run 'cautious-badger' - Executing 'task_extract-0' immediately...

06:41:55.858 | INFO    | Task run 'extract-api-football-fixtures' - Finished in state Completed()

...

06:42:05.258 | INFO    | Flow run 'cautious-badger' - Created task run 'task_transform_gold-0' for task 'task_transform_gold'

06:42:05.261 | INFO    | Flow run 'cautious-badger' - Executing 'task_transform_gold-0' immediately...

06:42:07.359 | INFO    | Task run 'transform-gold-fixtures' - Finished in state Completed()

06:42:07.879 | INFO    | Flow run 'cautious-badger' - Finished in state Completed('All states completed.')

[Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `list`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `DataFrame`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `NoneType`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `DataFrame`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `DataFrame`'))]

### Opción B — Servir el flow como agente local (opcional)

Esta modalidad ejecuta el flujo de manera **continua**, permitiendo gestionarlo desde **Prefect Cloud** (ejecuciones, logs, programación y monitoreo).

⚠️ Al activarla, la celda permanecerá ocupada hasta detenerla manualmente (*Interrupt/Stop*).  
Por este motivo, la llamada queda **comentada por defecto** para evitar ejecuciones involuntarias.

Si se habilita, el flujo se ejecutará **una vez al día a las 06:00 UTC**, según la configuración definida en `scripts/etl_fixtures.py`.

In [ ]:
import importlib
etl = importlib.import_module("scripts.etl_fixtures")

# etl.etl_api_football.serve(
#     name="ETL-Fixtures",
#     endpoints=["fixtures"]
# )

## Monitoreo en Prefect

El monitoreo del flujo no se realiza desde el notebook, sino desde la **UI de Prefect Cloud**.  
Una vez que el flow se ejecuta (corrida única o servido como agente), es posible:

1. **Flows** → ver el listado de flows registrados (ejemplo: `ETL-Fixtures`).  
2. **Run history** → revisar el historial de ejecuciones con sus logs, tiempos y reintentos.  
3. **Blocks** → administrar la configuración de almacenamiento o infraestructura remota si se utiliza.  
4. **Schedules** → programar ejecuciones automáticas (requiere plan pago).